# 🎮 Day 23 — Deep RL, DQN Concepts, Multi-Agent & Safe RL
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Q-Learning Deep Dive & DQN Architecture | ~12 sec |
| 2 | 10:30–11:00 AM | Multi-Agent RL | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Safe RL + Constrained MDPs + Actor-Critic | ~18 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | RL Capstone: Train a Policy from Scratch | ~20 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, time
from collections import deque
warnings.filterwarnings('ignore')

from sklearn.datasets import load_iris, load_wine, load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — Q-Learning Deep Dive & DQN Architecture
**Time:** 9:00–10:30 AM | **Run time: ~12 sec**

### Bellman Equation
```
Q(s,a) ← Q(s,a) + α · [r + γ · max_a' Q(s',a') − Q(s,a)]
```

In [ ]:
# 1.1  GridWorld Environment
class GridWorld:
    """
    5×5 grid. Agent starts at (0,0), goal at (4,4).
    Actions: 0=Up, 1=Right, 2=Down, 3=Left
    Reward: +1.0 on reaching goal, -0.01 each step, -0.5 hitting wall
    """
    def __init__(self, size=5):
        self.size  = size
        self.n_states  = size * size
        self.n_actions = 4
        self.goal  = size*size - 1
        self.MOVES = [(-1,0),(0,1),(1,0),(0,-1)]  # Up Right Down Left

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        row, col = divmod(self.state, self.size)
        dr, dc   = self.MOVES[action]
        nr, nc   = row+dr, col+dc
        if 0 <= nr < self.size and 0 <= nc < self.size:
            self.state = nr * self.size + nc
            reward     = 1.0 if self.state == self.goal else -0.01
        else:
            reward = -0.3   # wall penalty (stay in place)
        done = (self.state == self.goal)
        return self.state, reward, done

env = GridWorld(size=5)
print(f'GridWorld: {env.n_states} states, {env.n_actions} actions, goal={env.goal}')
s = env.reset()
for a, name in zip([1,1,2,2],["Right","Right","Down","Down"]):
    ns, r, done = env.step(a)
    print(f'  Action={name:5s} → state={ns:2d}  reward={r:+.2f}  done={done}')

In [ ]:
# 1.2  Tabular Q-Learning with ε-Greedy Exploration
rng  = np.random.default_rng(42)
env  = GridWorld(size=5)
Q    = np.zeros((env.n_states, env.n_actions))

ALPHA   = 0.1    # learning rate
GAMMA   = 0.95   # discount factor
EPS     = 1.0    # initial exploration
EPS_MIN = 0.05
EPS_DECAY = 0.995
N_EPISODES = 800

episode_rewards = []
episode_lengths = []

for ep in range(N_EPISODES):
    s    = env.reset()
    total_r = 0
    steps   = 0
    EPS     = max(EPS_MIN, EPS * EPS_DECAY)

    for _ in range(100):  # max 100 steps per episode
        # ε-greedy action selection
        if rng.random() < EPS:
            a = rng.integers(env.n_actions)
        else:
            a = Q[s].argmax()

        ns, r, done = env.step(a)

        # Bellman update
        Q[s, a] += ALPHA * (r + GAMMA * Q[ns].max() - Q[s, a])
        s        = ns
        total_r += r
        steps   += 1
        if done:
            break

    episode_rewards.append(total_r)
    episode_lengths.append(steps)

print(f'Q-Learning Training ({N_EPISODES} episodes):')
print(f'  Final ε          : {EPS:.4f}')
print(f'  Last 50ep reward : {np.mean(episode_rewards[-50:]):.4f}')
print(f'  Last 50ep length : {np.mean(episode_lengths[-50:]):.1f} steps')
print(f'\nLearned Q-table (first 5 states):')
for s in range(5):
    row,col = divmod(s,5)
    best_a  = Q[s].argmax()
    print(f'  State {s} ({row},{col}): Q={Q[s].round(3)}  best={["Up","Right","Down","Left"][best_a]}')

In [ ]:
# 1.3  Experience Replay Buffer
class ReplayBuffer:
    """
    Store (s, a, r, s', done) transitions.
    Sample random mini-batches to break temporal correlation.
    This is the key innovation of DQN over tabular Q-learning.
    """
    def __init__(self, capacity=10_000):
        self.buffer   = deque(maxlen=capacity)
        self.capacity = capacity

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        idx     = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch   = [self.buffer[i] for i in idx]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states),   np.array(actions),
                np.array(rewards),  np.array(next_states),
                np.array(dones, dtype=float))

    def __len__(self):
        return len(self.buffer)

# Fill buffer with random experience
buf = ReplayBuffer(capacity=5000)
env2 = GridWorld(size=5)
s2   = env2.reset()
for _ in range(2000):
    a2       = rng.integers(env2.n_actions)
    ns2,r2,done2 = env2.step(a2)
    buf.push(s2, a2, r2, ns2, done2)
    s2       = env2.reset() if done2 else ns2

states, actions, rewards, next_states, dones = buf.sample(32)
print(f'Experience Replay Buffer:')
print(f'  Total stored     : {len(buf):,}')
print(f'  Batch states     : {states.shape}  (32 random transitions)')
print(f'  Reward stats     : mean={rewards.mean():.4f}  std={rewards.std():.4f}')
print(f'  Done fraction    : {dones.mean():.3f}')

In [ ]:
# 1.4  Target Network Concept + Q-Learning with Replay
class LinearQNetwork:
    """Simple linear Q-function approximator: Q(s,a) = W[s] (table-based for speed)."""
    def __init__(self, n_states, n_actions, seed=42):
        rng_q = np.random.default_rng(seed)
        self.W = rng_q.normal(0, 0.01, (n_states, n_actions))

    def predict(self, states):
        return self.W[states]

    def update(self, states, actions, targets, lr=0.1):
        for s, a, t in zip(states, actions, targets):
            self.W[s, a] += lr * (t - self.W[s, a])

    def copy_weights_from(self, other):
        self.W = other.W.copy()

# DQN-style training with replay + target network
online_net = LinearQNetwork(env.n_states, env.n_actions)
target_net = LinearQNetwork(env.n_states, env.n_actions)
target_net.copy_weights_from(online_net)

TARGET_UPDATE = 20   # update target every N episodes
BATCH_SIZE    = 32
EPS2          = 1.0
env3          = GridWorld(size=5)
buf2          = ReplayBuffer(5000)
dqn_rewards   = []

for ep in range(600):
    s3 = env3.reset(); total = 0
    EPS2 = max(0.05, EPS2 * 0.995)
    for _ in range(100):
        a3 = rng.integers(env3.n_actions) if rng.random()<EPS2 else online_net.predict(np.array([s3])).argmax()
        ns3, r3, done3 = env3.step(a3)
        buf2.push(s3, a3, r3, ns3, done3)
        s3 = env3.reset() if done3 else ns3
        total += r3
        # Train on mini-batch
        if len(buf2) >= BATCH_SIZE:
            sb,ab,rb,nsb,db = buf2.sample(BATCH_SIZE)
            # Bellman targets using frozen target network
            q_next   = target_net.predict(nsb).max(axis=1)
            td_targets = rb + GAMMA*(1-db)*q_next
            online_net.update(sb, ab, td_targets)
        if done3: break
    dqn_rewards.append(total)
    if ep % TARGET_UPDATE == 0:
        target_net.copy_weights_from(online_net)

print(f'DQN with Replay + Target Network ({600} episodes):')
print(f'  Last 50ep avg reward: {np.mean(dqn_rewards[-50:]):.4f}')
print(f'  Target updates done : {600 // TARGET_UPDATE}')

In [ ]:
# 1.5  Learning Curves Comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Q-learning curve
window = 30
q_smooth = pd.Series(episode_rewards).rolling(window).mean()
axes[0].plot(q_smooth, color='#534AB7', linewidth=1.8)
axes[0].set_title('Tabular Q-Learning')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel(f'Reward (rolling {window})')
axes[0].grid(alpha=0.3)

# DQN with replay
dqn_smooth = pd.Series(dqn_rewards).rolling(window).mean()
axes[1].plot(dqn_smooth, color='#1D9E75', linewidth=1.8)
axes[1].set_title('DQN (Replay + Target Network)')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel(f'Reward (rolling {window})')
axes[1].grid(alpha=0.3)

# Optimal policy visualisation
policy_grid = np.array([Q[s].argmax() for s in range(25)]).reshape(5,5)
SYM = ['↑','→','↓','←']
policy_str = np.array([[SYM[policy_grid[r,c]] for c in range(5)] for r in range(5)])
im = axes[2].imshow(policy_grid, cmap='RdBu', vmin=0, vmax=3)
for r in range(5):
    for c in range(5):
        color = 'gold' if r*5+c == env.goal else 'white'
        axes[2].text(c, r, policy_str[r,c], ha='center', va='center', fontsize=16, color=color)
axes[2].set_title('Learned Policy (↑→↓←, gold=goal)')
axes[2].set_xticks([]); axes[2].set_yticks([])
plt.suptitle('Q-Learning & DQN Learning Curves', fontsize=12)
plt.tight_layout(); plt.show()

---
## Session 2 — Multi-Agent RL
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  Cooperative Multi-Agent Q-Learning
rng_ma = np.random.default_rng(42)
N_AGENTS  = 2
N_STATES  = 16   # 4×4 grid
N_ACTIONS = 4    # Up Right Down Left
GOAL_MA   = 15   # bottom-right corner

# Each agent has its own Q-table
Q_agents = [np.zeros((N_STATES, N_ACTIONS)) for _ in range(N_AGENTS)]
LR_MA = 0.15; GAMMA_MA = 0.9
MA_MOVES = [(-1,0),(0,1),(1,0),(0,-1)]
GRID_SZ  = 4

def ma_step(state, action, grid_sz=4):
    r, c  = divmod(state, grid_sz)
    dr,dc = MA_MOVES[action]
    nr,nc = r+dr, c+dc
    if 0 <= nr < grid_sz and 0 <= nc < grid_sz:
        ns = nr*grid_sz + nc
    else:
        ns = state
    return ns

ma_rewards_per_ep = []

for ep in range(500):
    states    = [0, 3]   # agents start at different corners
    total_r   = 0
    eps_ma    = max(0.05, 1.0 - ep/400)
    for _ in range(50):
        actions = []
        for i in range(N_AGENTS):
            if rng_ma.random() < eps_ma:
                actions.append(rng_ma.integers(N_ACTIONS))
            else:
                actions.append(Q_agents[i][states[i]].argmax())

        next_states = [ma_step(states[i], actions[i]) for i in range(N_AGENTS)]
        # Joint reward: +2 if both at goal, else step penalty
        if all(ns == GOAL_MA for ns in next_states):
            r_joint = 2.0
        elif any(ns == GOAL_MA for ns in next_states):
            r_joint = 0.5   # partial reward if one arrives
        else:
            r_joint = -0.01
        total_r += r_joint

        for i in range(N_AGENTS):
            td = r_joint + GAMMA_MA * Q_agents[i][next_states[i]].max() - Q_agents[i][states[i], actions[i]]
            Q_agents[i][states[i], actions[i]] += LR_MA * td

        states = next_states
        if r_joint == 2.0:
            break

    ma_rewards_per_ep.append(total_r)

SYM_MA = ['↑','→','↓','←']
print('Cooperative Multi-Agent Q-Learning (2 agents, 4×4 grid, goal=15):')
for i in range(N_AGENTS):
    print(f'  Agent {i} start policy: {[SYM_MA[Q_agents[i][s].argmax()] for s in [0,3,5,10,15]]}')
print(f'  Last 50ep joint reward: {np.mean(ma_rewards_per_ep[-50:]):.4f}')

In [ ]:
# 2.2  Nash Equilibrium in a Simple Matrix Game
# Rock-Paper-Scissors: payoff matrix for Player 1
ROCK,PAPER,SCISSORS = 0,1,2

# Payoff: +1 win, -1 lose, 0 draw
PAYOFF = np.array([
    [ 0, -1,  1],   # Rock vs Rock/Paper/Scissors
    [ 1,  0, -1],   # Paper
    [-1,  1,  0],   # Scissors
])

def rps_best_response(opp_strategy):
    """Best response given opponent's mixed strategy."""
    expected = PAYOFF @ opp_strategy
    return expected.argmax()

def fictitious_play(n_iters=1000, seed=42):
    """Fictitious play converges to Nash equilibrium in zero-sum games."""
    rng_fp = np.random.default_rng(seed)
    counts = [np.ones(3), np.ones(3)]  # start with uniform
    history = []
    for _ in range(n_iters):
        strats = [c/c.sum() for c in counts]
        actions = [rps_best_response(strats[1-i]) for i in range(2)]
        for i in range(2):
            counts[i][actions[i]] += 1
        history.append([c/c.sum() for c in counts])
    return [c/c.sum() for c in counts], history

nash_strats, history = fictitious_play(n_iters=2000)
print('Fictitious Play → Nash Equilibrium in Rock-Paper-Scissors:')
for i, s in enumerate(nash_strats):
    print(f'  Player {i}: Rock={s[0]:.4f}  Paper={s[1]:.4f}  Scissors={s[2]:.4f}')
print(f'\nNash equilibrium = [1/3, 1/3, 1/3] ≈ {[round(x,4) for x in nash_strats[0]]}')

---
## Session 3 — Safe RL + Constrained MDPs + Actor-Critic
**Time:** 11:00 AM–1:00 PM | **Run time: ~18 sec**

In [ ]:
# 3.1  Constrained MDP (CMDP): Safe Exploration
rng_safe = np.random.default_rng(42)

# 5-action bandit with both reward and cost
N_ACTIONS_SAFE = 5
REWARDS = np.array([0.5, 0.8, 0.2, 0.9, 0.3])  # action rewards
COSTS   = np.array([0.1, 0.7, 0.1, 0.8, 0.2])  # action costs (higher-reward = riskier)
COST_LIMIT = 0.3   # expected cost must stay below this

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

# Unconstrained policy gradient
logits_unc = np.zeros(N_ACTIONS_SAFE)
LR_UNC = 0.05
unc_rewards, unc_costs = [], []
for step in range(300):
    probs = softmax(logits_unc)
    r_exp = probs @ REWARDS
    c_exp = probs @ COSTS
    # Vanilla policy gradient (maximise reward only)
    grad  = REWARDS - r_exp
    logits_unc += LR_UNC * grad
    unc_rewards.append(r_exp)
    unc_costs.append(c_exp)

# Lagrangian constrained policy gradient
logits_con = np.zeros(N_ACTIONS_SAFE)
lam        = 1.0   # Lagrange multiplier
LR_PI      = 0.05
LR_LAM     = 0.02
con_rewards, con_costs = [], []
for step in range(300):
    probs = softmax(logits_con)
    r_exp = probs @ REWARDS
    c_exp = probs @ COSTS
    # Constrained gradient: maximise reward - λ*cost
    grad_pi = REWARDS - lam*COSTS - (r_exp - lam*c_exp)
    logits_con += LR_PI * grad_pi
    # Dual update: increase λ if constraint violated
    lam = max(0, lam + LR_LAM * (c_exp - COST_LIMIT))
    con_rewards.append(r_exp)
    con_costs.append(c_exp)

print('Constrained MDP — Lagrangian Safe RL:')
p_unc = softmax(logits_unc)
p_con = softmax(logits_con)
print(f'  Unconstrained : reward={p_unc@REWARDS:.4f}  cost={p_unc@COSTS:.4f}  SAFE={p_unc@COSTS<=COST_LIMIT}')
print(f'  Constrained   : reward={p_con@REWARDS:.4f}  cost={p_con@COSTS:.4f}  SAFE={p_con@COSTS<=COST_LIMIT}')
print(f'  Final λ       : {lam:.4f}  (higher = constraint was active)')

In [ ]:
# 3.2  Actor-Critic on Custom Classification MDP
rng_ac = np.random.default_rng(42)

X_ac, y_ac = load_wine(return_X_y=True)
sc_ac  = StandardScaler().fit(X_ac)
X_ac_n = sc_ac.transform(X_ac)
D_AC   = X_ac_n.shape[1]
K_AC   = 3   # classes = actions

# Actor weights (policy)
W_actor  = rng_ac.normal(0, 0.05, (D_AC, K_AC))
# Critic weights (value)
W_critic = rng_ac.normal(0, 0.05, D_AC)

LR_A = 0.005; LR_C = 0.01; GAMMA_AC = 0.95
ac_rewards, ac_accs = [], []

for ep in range(300):
    idx   = rng_ac.integers(len(X_ac_n))
    x, y  = X_ac_n[idx], y_ac[idx]

    # Actor: compute policy
    logits_ac = x @ W_actor
    probs_ac  = softmax(logits_ac)
    action    = rng_ac.choice(K_AC, p=probs_ac)
    reward    = 1.0 if action == y else -0.3

    # Critic: estimate value baseline
    value     = x @ W_critic
    advantage = reward - value   # TD(0) advantage

    # Critic update: minimise (reward - value)^2
    W_critic += LR_C * advantage * x

    # Actor update: policy gradient with advantage
    grad_log    = np.zeros(K_AC)
    grad_log[action] = 1 - probs_ac[action]   # ∂log π / ∂logits
    grad_log    -= -probs_ac * probs_ac[action]
    W_actor     += LR_A * advantage * np.outer(x, grad_log)

    ac_rewards.append(reward)
    ac_accs.append(int(action == y))

print('Actor-Critic on Wine Classification MDP:')
print(f'  Final 50ep accuracy  : {np.mean(ac_accs[-50:]):.4f}')
print(f'  Final 50ep reward    : {np.mean(ac_rewards[-50:]):.4f}')
print(f'  Initial 50ep accuracy: {np.mean(ac_accs[:50]):.4f}')

In [ ]:
# 3.3  Visualise Safe RL + Actor-Critic
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Constrained vs unconstrained
steps = range(len(unc_rewards))
ax = axes[0]
ax.plot(unc_costs,  color='#D85A30', linewidth=1.5, alpha=0.8, label='Unconstrained cost')
ax.plot(con_costs,  color='#534AB7', linewidth=1.5, alpha=0.8, label='Constrained cost')
ax.axhline(COST_LIMIT, linestyle='--', color='#1D9E75', linewidth=2, label=f'Cost limit={COST_LIMIT}')
ax.set_title('Safe RL: Lagrangian Constraint')
ax.set_xlabel('Training step'); ax.set_ylabel('Expected cost')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax2 = axes[1]
ax2.plot(pd.Series(unc_rewards).rolling(20).mean(), color='#D85A30', linewidth=1.5, label='Unconstrained reward')
ax2.plot(pd.Series(con_rewards).rolling(20).mean(), color='#534AB7', linewidth=1.5, label='Constrained reward')
ax2.set_title('Safe RL: Reward Trade-Off')
ax2.set_xlabel('Training step'); ax2.set_ylabel('Expected reward')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

ax3 = axes[2]
ax3.plot(pd.Series(ac_accs).rolling(30).mean(), color='#1D9E75', linewidth=2)
ax3.set_title('Actor-Critic Accuracy (Wine)')
ax3.set_xlabel('Episode'); ax3.set_ylabel('Accuracy (rolling 30)')
ax3.grid(alpha=0.3)

plt.suptitle('Safe RL + Actor-Critic', fontsize=12)
plt.tight_layout(); plt.show()

---
## Lunch Break — 1:00–2:00 PM
1. Why does experience replay break temporal correlation in DQN?
2. Why do we need a frozen target network — what goes wrong without it?
3. What is the difference between cooperative and competitive multi-agent RL?
4. How does the Lagrange multiplier λ enforce a safety cost constraint?
---

## Session 5 — RL Capstone: Train a Policy from Scratch
**Time:** 2:00–4:00 PM | **Run time: ~20 sec**

In [ ]:
# 5.1  Custom MDP: Inventory Management
class InventoryMDP:
    """
    Agent manages stock of a product:
    - State: current inventory level (0..MAX_INV)
    - Action: order quantity (0..MAX_ORDER)
    - Reward: sales_profit - holding_cost - stockout_penalty
    - Demand: Poisson-distributed each step
    """
    def __init__(self, max_inv=20, max_order=10, demand_lambda=5, seed=42):
        self.max_inv    = max_inv
        self.max_order  = max_order
        self.lam        = demand_lambda
        self.rng        = np.random.default_rng(seed)
        self.n_states   = max_inv + 1
        self.n_actions  = max_order + 1
        self.inv        = max_inv // 2

    def reset(self):
        self.inv = self.max_inv // 2
        return self.inv

    def step(self, action):
        # Receive order
        self.inv = min(self.max_inv, self.inv + action)
        # Demand realised
        demand  = self.rng.poisson(self.lam)
        sold    = min(demand, self.inv)
        lost    = max(0, demand - self.inv)
        self.inv = max(0, self.inv - demand)
        # Reward
        reward = sold * 2.0 - self.inv * 0.5 - lost * 3.0 - action * 0.2
        return self.inv, reward, False  # never terminal

mdp = InventoryMDP(max_inv=20, max_order=10, demand_lambda=5)
print('Inventory MDP:')
print(f'  States  : {mdp.n_states} (inventory 0–20)')
print(f'  Actions : {mdp.n_actions} (order 0–10 units)')
s = mdp.reset()
for a, name in [(0,'order 0'),(5,'order 5'),(10,'order 10')]:
    mdp.reset()
    ns, r, _ = mdp.step(a)
    print(f'  {name}: inv after={ns}  reward={r:.3f}')

In [ ]:
# 5.2  Q-Learning on Inventory MDP
rng_inv = np.random.default_rng(42)
env_inv = InventoryMDP(seed=42)
Q_inv   = np.zeros((env_inv.n_states, env_inv.n_actions))

ALPHA_INV = 0.1; GAMMA_INV = 0.95; EPS_INV = 1.0
N_EPISODES_INV = 1000
inv_rewards = []

for ep in range(N_EPISODES_INV):
    s_inv  = env_inv.reset()
    total  = 0
    EPS_INV = max(0.05, EPS_INV * 0.997)
    for t in range(30):  # 30 steps per episode
        if rng_inv.random() < EPS_INV:
            a_inv = rng_inv.integers(env_inv.n_actions)
        else:
            a_inv = Q_inv[s_inv].argmax()
        ns_inv, r_inv, _ = env_inv.step(a_inv)
        Q_inv[s_inv, a_inv] += ALPHA_INV*(r_inv + GAMMA_INV*Q_inv[ns_inv].max() - Q_inv[s_inv, a_inv])
        s_inv = ns_inv; total += r_inv
    inv_rewards.append(total)

print('Q-Learning on Inventory MDP:')
print(f'  Last 100ep avg reward : {np.mean(inv_rewards[-100:]):.4f}')
print(f'  Initial 100ep avg     : {np.mean(inv_rewards[:100]):.4f}')
print(f'  Improvement           : {np.mean(inv_rewards[-100:])-np.mean(inv_rewards[:100]):+.4f}')

# Learned policy
print('\nLearned order policy (by inventory level):')
for inv_lvl in [0, 5, 10, 15, 20]:
    order = Q_inv[inv_lvl].argmax()
    print(f'  Inventory={inv_lvl:2d}: order {order} units')

In [ ]:
# 5.3  Safe Actor-Critic on Inventory MDP with Cost Constraint
# Cost: ordering too much (> 8 units) is 'unsafe' (supply chain risk)
rng_sac = np.random.default_rng(0)
env_sac = InventoryMDP(seed=0)

W_a = rng_sac.normal(0, 0.01, (env_sac.n_states, env_sac.n_actions))
W_c = rng_sac.normal(0, 0.01, env_sac.n_states)
W_cost = rng_sac.normal(0, 0.01, env_sac.n_states)  # cost critic
lam_sac   = 0.5
SAFE_LIMIT = 0.25  # max fraction of orders > 8 units allowed
LR_SAC_A = 0.008; LR_SAC_C = 0.015; LR_SAC_LAM = 0.01

sac_rewards, sac_costs = [], []

for ep in range(600):
    s_sac  = env_sac.reset()
    total_r = 0; total_c = 0
    for t in range(30):
        logits_sac = W_a[s_sac]
        probs_sac  = softmax(logits_sac)
        a_sac      = rng_sac.choice(env_sac.n_actions, p=probs_sac)
        ns_sac, r_sac, _ = env_sac.step(a_sac)
        cost = 1.0 if a_sac > 8 else 0.0   # ordering > 8 = unsafe

        # Advantage
        v   = W_c[s_sac]
        cv  = W_cost[s_sac]
        adv = r_sac - lam_sac*cost - v + lam_sac*cv
        # Update critics
        W_c[s_sac]    += LR_SAC_C   * (r_sac - v)
        W_cost[s_sac] += LR_SAC_C   * (cost  - cv)
        # Update actor
        g_log = np.zeros(env_sac.n_actions)
        g_log[a_sac] = 1 - probs_sac[a_sac]
        g_log -= -probs_sac * probs_sac[a_sac]
        W_a[s_sac] += LR_SAC_A * adv * g_log
        # Dual update
        lam_sac = max(0, lam_sac + LR_SAC_LAM*(cost - SAFE_LIMIT))
        s_sac = ns_sac; total_r += r_sac; total_c += cost
    sac_rewards.append(total_r/30); sac_costs.append(total_c/30)

print('Safe Actor-Critic (Inventory MDP):')
print(f'  Last 100ep avg reward      : {np.mean(sac_rewards[-100:]):.4f}')
print(f'  Last 100ep avg cost rate   : {np.mean(sac_costs[-100:]):.4f}  (limit={SAFE_LIMIT})')
print(f'  Constraint satisfied       : {np.mean(sac_costs[-100:]) <= SAFE_LIMIT}')
print(f'  Final λ                    : {lam_sac:.4f}')

In [ ]:
# 5.4  Curriculum Learning: Progressively Harder Gridworld
class CurriculumGridWorld:
    """Start with 3×3, grow to 5×5, then 7×7."""
    def __init__(self, size=3):
        self.size     = size
        self.n_states = size * size
        self.n_actions= 4
        self.goal     = size*size - 1
        self.MOVES    = [(-1,0),(0,1),(1,0),(0,-1)]
        self.state    = 0

    def reset(self):
        self.state = 0; return 0

    def step(self, action):
        r, c    = divmod(self.state, self.size)
        dr, dc  = self.MOVES[action]
        nr, nc  = r+dr, c+dc
        if 0<=nr<self.size and 0<=nc<self.size:
            self.state = nr*self.size+nc
            reward     = 1.0 if self.state==self.goal else -0.01
        else:
            reward = -0.2
        return self.state, reward, (self.state==self.goal)

rng_cur = np.random.default_rng(42)
curriculum_results = {}

for size, n_ep in [(3,200),(5,400),(7,600)]:
    env_cur = CurriculumGridWorld(size=size)
    Q_cur   = np.zeros((env_cur.n_states, env_cur.n_actions))
    eps_c   = 1.0
    rewards_c = []
    for ep in range(n_ep):
        s_c = env_cur.reset(); tot = 0
        eps_c = max(0.05, eps_c*0.995)
        for _ in range(size*size*4):
            a_c = rng_cur.integers(env_cur.n_actions) if rng_cur.random()<eps_c else Q_cur[s_c].argmax()
            ns_c,r_c,done_c = env_cur.step(a_c)
            Q_cur[s_c,a_c] += 0.1*(r_c+0.95*Q_cur[ns_c].max()-Q_cur[s_c,a_c])
            s_c = ns_c; tot += r_c
            if done_c: break
        rewards_c.append(tot)
    curriculum_results[size] = np.mean(rewards_c[-50:])
    print(f'{size}×{size} Grid ({n_ep} eps): last-50ep avg reward = {curriculum_results[size]:.4f}')

print('\nCurriculum: agent mastered progressively harder environments!')

In [ ]:
# 5.5  Final Capstone Visualisation
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Inventory Q-learning convergence
axes[0,0].plot(pd.Series(inv_rewards).rolling(50).mean(), color='#534AB7', linewidth=2)
axes[0,0].set_title('Inventory Q-Learning Convergence')
axes[0,0].set_xlabel('Episode'); axes[0,0].set_ylabel('Episode reward')
axes[0,0].grid(alpha=0.3)

# Safe AC: reward vs cost
axes[0,1].plot(pd.Series(sac_rewards).rolling(30).mean(), color='#1D9E75', linewidth=2, label='Reward/step')
axes[0,1].plot(pd.Series(sac_costs).rolling(30).mean(),   color='#D85A30', linewidth=2, label='Cost rate')
axes[0,1].axhline(SAFE_LIMIT, linestyle='--', color='black', linewidth=1.5, label='Safety limit')
axes[0,1].set_title('Safe Actor-Critic (Inventory)')
axes[0,1].set_xlabel('Episode'); axes[0,1].legend(fontsize=8); axes[0,1].grid(alpha=0.3)

# Multi-agent rewards
axes[1,0].plot(pd.Series(ma_rewards_per_ep).rolling(30).mean(), color='#D85A30', linewidth=2)
axes[1,0].set_title('Multi-Agent Cooperative Q-Learning')
axes[1,0].set_xlabel('Episode'); axes[1,0].set_ylabel('Joint episode reward')
axes[1,0].grid(alpha=0.3)

# Curriculum results
sizes   = list(curriculum_results.keys())
results = list(curriculum_results.values())
bars = axes[1,1].bar([f'{s}×{s}' for s in sizes], results, color=[C for C in ['#534AB7','#1D9E75','#D85A30']], alpha=0.85)
axes[1,1].bar_label(bars, fmt='%.3f', padding=4)
axes[1,1].set_title('Curriculum Learning: Avg Reward by Grid Size')
axes[1,1].set_ylabel('Last-50ep avg reward')
axes[1,1].grid(axis='y', alpha=0.3)

plt.suptitle('Day 23 RL Capstone: Q-Learning + Safe AC + Multi-Agent + Curriculum', fontsize=11)
plt.tight_layout(); plt.show()

print('\n=== DAY 23 RL CAPSTONE REPORT ===')
print(f'  Tabular Q-Learning (GridWorld) : last-50ep reward = {np.mean(episode_rewards[-50:]):.4f}')
print(f'  DQN w/ Replay + Target Net     : last-50ep reward = {np.mean(dqn_rewards[-50:]):.4f}')
print(f'  Multi-Agent Cooperative        : last-50ep reward = {np.mean(ma_rewards_per_ep[-50:]):.4f}')
print(f'  Actor-Critic (Wine)            : last-50ep accuracy= {np.mean(ac_accs[-50:]):.4f}')
print(f'  Safe Actor-Critic (Inventory)  : cost rate = {np.mean(sac_costs[-100:]):.4f} ≤ {SAFE_LIMIT}')
print(f'  Curriculum (3→5→7 Grid)        : {" → ".join([str(round(v,3)) for v in curriculum_results.values()])}')

---
## Day 23 Summary

| Concept | Skill Gained |
|---|---|
| Bellman equation | Q(s,a) ← Q + α[r + γ·maxQ' - Q] from scratch |
| ε-greedy decay | Exploration → exploitation schedule |
| Experience replay | Random mini-batch breaks temporal correlation |
| Target network | Frozen copy stabilises TD targets |
| Cooperative MARL | Joint reward, independent Q-tables per agent |
| Nash equilibrium | Fictitious play convergence in RPS |
| CMDP | Constrained MDP: max reward subject to E[cost] ≤ limit |
| Lagrangian RL | Dual variable λ enforces safety constraint |
| Actor-Critic | Actor policy + critic value baseline |
| Custom MDP (Inventory) | Design state/action/reward, train Q-table |
| Safe AC | Cost critic + Lagrangian dual update |
| Curriculum learning | Progressive environment difficulty scaling |

---
## Day 24 Preview
- Transformer architecture: full implementation from scratch
- Positional encoding and multi-head attention revisited
- Sequence-to-sequence learning and encoder-decoder
- Language model fine-tuning patterns
- Capstone: build a mini transformer for sequence classification

In [ ]:
# Bonus: Day 23 in one cell
rng_b = np.random.default_rng(0)
env_b = GridWorld(size=4)
Q_b   = np.zeros((16, 4))
eps_b = 1.0
rew_b = []
for ep in range(400):
    s_b = env_b.reset(); tot = 0; eps_b = max(0.05, eps_b*0.997)
    for _ in range(60):
        a_b = rng_b.integers(4) if rng_b.random()<eps_b else Q_b[s_b].argmax()
        ns_b,r_b,done_b = env_b.step(a_b)
        Q_b[s_b,a_b] += 0.1*(r_b+0.95*Q_b[ns_b].max()-Q_b[s_b,a_b])
        s_b=ns_b; tot+=r_b
        if done_b: break
    rew_b.append(tot)
print(f'Q-Learning 4x4: last-50ep reward = {np.mean(rew_b[-50:]):.4f}')

# Safe RL check
probs_b = softmax(logits_con)
print(f'Safe policy: reward={probs_b@REWARDS:.4f}  cost={probs_b@COSTS:.4f} (limit {COST_LIMIT})')
print(f'\nDay 23 complete — Q-Learning, DQN, Multi-Agent, Safe RL, Actor-Critic, Curriculum!')